# Session 3, Part B — Recursion & Recursive Thinking

Read each cell, **predict** the output, then run it with **Shift + Enter**. The practice (solutions collapsed) is at the end.

**Tips:** press **Tab** to autocomplete a name, **Shift + Tab** for a function's help. Need a library? Run `%pip install <name>` in a cell (e.g. `%pip install pandas`) — in the browser (JupyterLite) it fetches a Pyodide build and lasts for the session.

In [ ]:
import functools
import sys

In [ ]:
# 1) The shape of EVERY recursion: a BASE CASE that stops + a RECURSIVE CASE
#    that moves toward it. Real example: how many prerequisites deep is a course?
prereq_of = {
    "ED700": "ED600",    # to take ED700 you must first pass ED600,
    "ED600": "ED500",    #   which needs ED500,
    "ED500": "ED400",    #   which needs ED400,
    "ED400": None,       #   which has no prerequisite — the base case
}

def prereqs_deep(course):
    earlier = prereq_of[course]
    if earlier is None:                 # BASE CASE — nothing comes before it
        return 0
    return 1 + prereqs_deep(earlier)    # RECURSIVE CASE — step back one course

print("ED700 prerequisite depth:", prereqs_deep("ED700"))   # 3

In [ ]:
# 2) Recursion vs iteration: how many distinct ways can n students finish a
#    race? That is n! (n factorial). Same answer two ways; the loop is clearer.
def orderings(n):
    if n <= 1:                          # base case: 0 or 1 student -> one order
        return 1
    return n * orderings(n - 1)         # <- you MUST return the recursive call

def orderings_loop(n):
    total = 1
    for k in range(2, n + 1):
        total *= k
    return total

print("\nways to rank 5 students:", orderings(5), "==", orderings_loop(5))   # 120

In [ ]:
# 3) Memoization: @lru_cache remembers past results so each input is computed
#    once. A value built from smaller copies of itself — here a growth model,
#    the Fibonacci sequence — otherwise recomputes the same subtotals exponentially.
calls = 0
def growth_naive(week):
    global calls
    calls += 1
    return week if week < 2 else growth_naive(week - 1) + growth_naive(week - 2)

@functools.lru_cache(maxsize=None)      # one line turns the slow version fast
def growth_fast(week):
    return week if week < 2 else growth_fast(week - 1) + growth_fast(week - 2)

print("\ngrowth_naive(30):", growth_naive(30), "in", calls, "calls")
print("growth_fast(30): ", growth_fast(30), "->", growth_fast.cache_info())
# The same @lru_cache speeds up ANY expensive repeated call — a slow file
# parse, a web lookup, a database query.

In [ ]:
# 4) Where recursion SHINES: naturally NESTED data, where one loop can't reach
#    all the way down. This is shaped like a real nested-JSON survey export.
export = {
    "cohort": "2026",
    "students": [
        {"name": "Ana", "scores": [91, 88]},
        {"name": "Ben", "scores": [58, [60, 64]]},   # arbitrarily nested
    ],
}

def deep_sum(obj):
    """Add up every number found anywhere inside nested lists/dicts."""
    if isinstance(obj, bool):                 # bool is an int subclass (Session 1!)
        return 0
    if isinstance(obj, (int, float)):
        return obj
    if isinstance(obj, dict):
        return sum(deep_sum(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return sum(deep_sum(x) for x in obj)
    return 0                                  # strings, None, etc. contribute nothing

print("\ndeep_sum of nested export:", deep_sum(export))   # 91+88+58+60+64 = 361

In [ ]:
# 5) The trap: with no reachable base case, recursion never stops. Python has no
#    tail-call optimization, so it just piles up stack frames until it gives up.
print("\nPython's recursion limit:", sys.getrecursionlimit())

def runaway(n):
    return runaway(n + 1)        # BUG: never reaches a base case

try:
    runaway(0)
except RecursionError:
    print("RecursionError: maximum recursion depth exceeded (as expected)")

## Now you try — Practice

### Task 1 — Recursive total
Write `total(scores)` that sums a list of scores **with recursion** (no loop):
`scores[0] + total(rest)`, with the empty list as the base case. Name the base
case out loud first. Test `total([91, 58, 73])` and `total([])`.

### Task 2 — Recursion vs iteration
Write `reverse(s)` that reverses a string recursively. Then write the loop version.
Which reads more clearly to you? Test `reverse("data")`.

### Task 3 — Flatten nested data
Write `flatten(xs)` that turns a list-of-lists (nested to any depth) into one flat list:
`flatten([1, [2, [3, 4]], 5])` → `[1, 2, 3, 4, 5]`. This is the move for nested JSON/exports.

### Task 4 — How deep does it go?
Write `depth(xs)` returning how deeply a list is nested:
`depth([1, [2, [3, [4]]]])` → `4`, `depth([1, 2, 3])` → `1`, `depth(5)` → `0`.

### Task 5 — Trap check
1. Why does this raise `RecursionError`, and what's the fix?
   ```python
   def f(n):
       return n + f(n - 1)
   ```
2. This returns `None` instead of a number — why?
   ```python
   def orderings(n):
       if n <= 1:
           return 1
       n * orderings(n - 1)
   ```
3. Name one case where a plain loop is the better choice over recursion.

### Bonus — Pythonic idiom drill
One decorator makes exponential recursion instant by remembering past calls.

```python
import functools

@functools.cache                     # memoize: each n is computed once
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(35))                       # -> 9227465   (try this WITHOUT @cache... then wait)
```

In [ ]:
# Your practice work — type here. Predict before you run.


<details>
<summary><strong>Show Practice solutions</strong></summary>

```python
# 1
def total(scores):
    if not scores:                  # base case: empty list sums to 0
        return 0
    return scores[0] + total(scores[1:])   # first score + total of the rest
print(total([91, 58, 73]), total([]))      # 222 0

# 2
def reverse(s):
    if s == "":                     # base case: empty string
        return ""
    return reverse(s[1:]) + s[0]    # all-but-first, reversed, then first
print(reverse("data"))             # "atad"
# loop version: "".join(reversed(s))  — usually clearer for flat strings

# 3
def flatten(xs):
    out = []
    for x in xs:
        if isinstance(x, list):
            out.extend(flatten(x))  # recurse into the sub-list
        else:
            out.append(x)
    return out
print(flatten([1, [2, [3, 4]], 5]))   # [1, 2, 3, 4, 5]

# 4
def depth(xs):
    if not isinstance(xs, list):
        return 0                              # a non-list has no nesting
    return 1 + max((depth(x) for x in xs), default=0)
print(depth([1, [2, [3, [4]]]]), depth([1, 2, 3]), depth(5))   # 4 1 0

# 5
# 1) No reachable base case -> the calls never stop -> stack overflows.
#    Fix: add `if n == 0: return 0` (or n <= 0) at the top.
# 2) The recursive case computes n*fact(n-1) but never RETURNs it,
#    so the function falls off the end and returns None. Add `return`.
# 3) A loop is better when the work is a simple flat sequence, or when the
#    depth could exceed ~1000 (Python has no tail-call optimization, so deep
#    recursion hits RecursionError where a loop would be fine).
```

</details>